# 04 BM25 与 Sparse Retrieval

## 学习目标

1. 理解 TF、DF、IDF
2. 理解 BM25 评分机制
3. 掌握中文分词
4. 使用 rank-bm25 构建关键词检索器
5. 对比 BM25 与 Dense Retrieval
6. 使用相同 Ground Truth 进行检索评测

In [1]:
documents = [
    "持续降雨会增加土体含水率，并可能提高孔隙水压力。",
    "地下水位升高可能降低边坡稳定性。",
    "坡度是影响滑坡发生的重要地形因素。",
    "太阳辐射是影响光伏发电功率的重要因素。",
    "高温可能降低光伏组件的转换效率。",
    "WRF 4.5.1 是一种数值天气预报模型版本。",
    "ERA5-Land 提供陆面气象再分析数据。",
    "RMSE 和 MAE 是常见的预测误差评价指标。",
    "PyTorch 可以用于深度学习模型训练。",
    "BGE-M3 支持多语言文本检索。",
    "Qdrant 是支持向量检索的数据库。",
    "Noah-MP 是一种陆面过程模型。"
]

In [2]:
import jieba

text = "持续降雨会增加土体含水率，并可能提高孔隙水压力。"

tokens = list(jieba.cut(text))

print(tokens)

import re
import jieba

def tokenize(text):
    # 统一英文字母大小写
    text = text.lower()

    # 保留专业缩写、数字和中文
    tokens = jieba.lcut(text)

    return [
        token.strip()
        for token in tokens
        if token.strip()
        and not re.fullmatch(r"\W+", token)
    ]

print(tokenize("持续降雨会不会引发滑坡？"))
print(tokenize("WRF 4.5.1 和 ERA5-Land 有什么区别？"))

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.603 seconds.
Prefix dict has been built successfully.


['持续', '降雨', '会', '增加', '土体', '含水率', '，', '并', '可能', '提高', '孔隙', '水', '压力', '。']
['持续', '降雨', '会', '不会', '引发', '滑坡']
['wrf', '4.5', '1', '和', 'era5', 'land', '有', '什么', '区别']


In [3]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    tokenize(doc)
    for doc in documents
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

In [4]:
query = "地下水位会不会影响边坡稳定性？"

query_tokens = tokenize(query)

scores = bm25.get_scores(query_tokens)

ranked_indices = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)

for rank, idx in enumerate(ranked_indices[:3], 1):
    print(f"Top {rank}")
    print(f"Score: {scores[idx]:.4f}")
    print(f"Text: {documents[idx]}")
    print("-" * 60)

Top 1
Score: 6.9391
Text: 地下水位升高可能降低边坡稳定性。
------------------------------------------------------------
Top 2
Score: 1.6817
Text: 持续降雨会增加土体含水率，并可能提高孔隙水压力。
------------------------------------------------------------
Top 3
Score: 1.3721
Text: 坡度是影响滑坡发生的重要地形因素。
------------------------------------------------------------


In [5]:
import numpy as np

def bm25_search(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked_ids = np.argsort(-scores)[:top_k]

    return [
        {
            "id": int(idx),
            "text": documents[idx],
            "score": float(scores[idx])
        }
        for idx in ranked_ids
    ]

results = bm25_search(
    "WRF 4.5.1 是什么？",
    top_k=3
)

for result in results:
    print(result)

{'id': 5, 'text': 'WRF 4.5.1 是一种数值天气预报模型版本。', 'score': 5.842373527948738}
{'id': 0, 'text': '持续降雨会增加土体含水率，并可能提高孔隙水压力。', 'score': 0.0}
{'id': 2, 'text': '坡度是影响滑坡发生的重要地形因素。', 'score': 0.0}


In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-zh-v1.5"
)
document_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

print(document_embeddings.shape)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

(12, 512)


In [7]:
def dense_search(query, top_k=3):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding

    ranked_ids = np.argsort(-scores)[:top_k]

    return [
        {
            "id": int(idx),
            "text": documents[idx],
            "score": float(scores[idx])
        }
        for idx in ranked_ids
    ]
for i, doc in enumerate(documents):
    print(f"{i}: {doc}")

evaluation_dataset = [
    {
        "query": "持续降雨会不会提高孔隙水压力？",
        "relevant_ids": [0],
        "type": "keyword"
    },
    {
        "query": "连续降水是否容易诱发坡体失稳？",
        "relevant_ids": [0, 1, 2],
        "type": "semantic"
    },
    {
        "query": "地下水位升高会影响边坡吗？",
        "relevant_ids": [1],
        "type": "semantic"
    },
    {
        "query": "影响太阳能发电的气象因素有哪些？",
        "relevant_ids": [3, 4],
        "type": "semantic"
    },
    {
        "query": "WRF 4.5.1 是什么？",
        "relevant_ids": [5],
        "type": "exact"
    },
    {
        "query": "ERA5-Land 是什么数据？",
        "relevant_ids": [6],
        "type": "exact"
    },
    {
        "query": "RMSE 和 MAE 用来评价什么？",
        "relevant_ids": [7],
        "type": "exact"
    },
    {
        "query": "Noah-MP 是什么模型？",
        "relevant_ids": [11],
        "type": "exact"
    }
]

0: 持续降雨会增加土体含水率，并可能提高孔隙水压力。
1: 地下水位升高可能降低边坡稳定性。
2: 坡度是影响滑坡发生的重要地形因素。
3: 太阳辐射是影响光伏发电功率的重要因素。
4: 高温可能降低光伏组件的转换效率。
5: WRF 4.5.1 是一种数值天气预报模型版本。
6: ERA5-Land 提供陆面气象再分析数据。
7: RMSE 和 MAE 是常见的预测误差评价指标。
8: PyTorch 可以用于深度学习模型训练。
9: BGE-M3 支持多语言文本检索。
10: Qdrant 是支持向量检索的数据库。
11: Noah-MP 是一种陆面过程模型。


In [8]:
def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    top_k_ids = set(retrieved_ids[:k])
    relevant_ids = set(relevant_ids)

    return int(bool(top_k_ids & relevant_ids))


def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    if not relevant_ids:
        return 0.0

    top_k_ids = set(retrieved_ids[:k])

    return len(top_k_ids & relevant_ids) / len(relevant_ids)


def reciprocal_rank_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    for rank, doc_id in enumerate(retrieved_ids[:k], 1):
        if doc_id in relevant_ids:
            return 1.0 / rank

    return 0.0

In [10]:
# 统一评价函数：
import pandas as pd
import time

def evaluate_search(
    dataset,
    search_func,
    method_name,
    top_k=3
):
    records = []

    for item in dataset:
        query = item["query"]
        relevant_ids = item["relevant_ids"]

        start = time.perf_counter()

        results = search_func(
            query,
            top_k=top_k
        )

        latency_ms = (
            time.perf_counter() - start
        ) * 1000

        retrieved_ids = [
            result["id"]
            for result in results
        ]

        records.append({
            "method": method_name,
            "type": item["type"],
            "query": query,
            "relevant_ids": relevant_ids,
            "retrieved_ids": retrieved_ids,
            "Hit@3": hit_rate_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "Recall@3": recall_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "RR@3": reciprocal_rank_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "latency_ms": latency_ms
        })

    return pd.DataFrame(records)

In [11]:
bm25_eval = evaluate_search(
    evaluation_dataset,
    bm25_search,
    method_name="BM25",
    top_k=3
)

dense_eval = evaluate_search(
    evaluation_dataset,
    dense_search,
    method_name="Dense",
    top_k=3
)

comparison_df = pd.concat(
    [bm25_eval, dense_eval],
    ignore_index=True
)

display(comparison_df)

,method,type,query,relevant_ids,retrieved_ids,Hit@3,Recall@3,RR@3,latency_ms
0,BM25,keyword,持续降雨会不会提高孔隙水压力？,[0],"[0, 1, 2]",1,1.0,1.0,0.4572
1,BM25,semantic,连续降水是否容易诱发坡体失稳？,"[0, 1, 2]","[0, 1, 2]",1,1.0,1.0,0.2613
2,BM25,semantic,地下水位升高会影响边坡吗？,[1],"[1, 0, 2]",1,1.0,1.0,0.1804
3,BM25,semantic,影响太阳能发电的气象因素有哪些？,"[3, 4]","[3, 2, 6]",1,0.5,1.0,0.2056
4,BM25,exact,WRF 4.5.1 是什么？,[5],"[5, 0, 2]",1,1.0,1.0,0.1492
5,BM25,exact,ERA5-Land 是什么数据？,[6],"[6, 0, 2]",1,1.0,1.0,0.1702
6,BM25,exact,RMSE 和 MAE 用来评价什么？,[7],"[7, 0, 2]",1,1.0,1.0,0.1880
7,BM25,exact,Noah-MP 是什么模型？,[11],"[11, 8, 5]",1,1.0,1.0,0.1145
8,Dense,keyword,持续降雨会不会提高孔隙水压力？,[0],"[0, 1, 4]",1,1.0,1.0,8.6103
9,Dense,semantic,连续降水是否容易诱发坡体失稳？,"[0, 1, 2]","[1, 0, 2]",1,1.0,1.0,7.9847


In [12]:
summary_df = (
    comparison_df
    .groupby("method")[
        ["Hit@3", "Recall@3", "RR@3", "latency_ms"]
    ]
    .mean()
    .reset_index()
    .rename(columns={"RR@3": "MRR@3"})
)

display(summary_df.round(4))

,method,Hit@3,Recall@3,MRR@3,latency_ms
0,BM25,1.0,0.9375,1.0,0.2158
1,Dense,1.0,1.0000,1.0,6.5125


In [16]:
type_comparison=(
    comparison_df
    .groupby(["type", "method"])[
        ["Hit@3", "Recall@3", "RR@3", "latency_ms"]
    ]
    .mean()
    .reset_index()
)
display(type_comparison.round(4))

,type,method,Hit@3,Recall@3,RR@3,latency_ms
0,exact,BM25,1.0,1.0000,1.0,0.1555
1,exact,Dense,1.0,1.0000,1.0,5.9702
2,keyword,BM25,1.0,1.0000,1.0,0.4572
3,keyword,Dense,1.0,1.0000,1.0,8.6103
4,semantic,BM25,1.0,0.8333,1.0,0.2158
5,semantic,Dense,1.0,1.0000,1.0,6.5364


In [17]:

# RRF
def reciprocal_rank_fusion(
    result_lists,
    rrf_k=60,
    top_k=3
):
    fused_scores = {}
    document_map = {}

    for results in result_lists:

        for rank, result in enumerate(results, start=1):

            doc_id = result["id"]

            fused_scores[doc_id] = (
                fused_scores.get(doc_id, 0.0)
                + 1.0 / (rrf_k + rank)
            )

            document_map[doc_id] = result["text"]

    ranked_ids = sorted(
        fused_scores,
        key=lambda doc_id: (-fused_scores[doc_id], doc_id)
    )

    return [
        {
            "id": doc_id,
            "text": document_map[doc_id],
            "score": fused_scores[doc_id]
        }
        for doc_id in ranked_ids[:top_k]
    ]

In [18]:
def hybrid_search(
    query,
    top_k=3,
    candidate_k=8,
    rrf_k=60
):
    bm25_results = bm25_search(
        query,
        top_k=candidate_k
    )

    dense_results = dense_search(
        query,
        top_k=candidate_k
    )

    fused_results = reciprocal_rank_fusion(
        result_lists=[
            bm25_results,
            dense_results
        ],
        rrf_k=rrf_k,
        top_k=top_k
    )

    return fused_results

In [19]:
query = "连续降水是否容易诱发坡体失稳？"

results = hybrid_search(
    query=query,
    top_k=3,
    candidate_k=8
)

for rank, result in enumerate(results, start=1):
    print("=" * 60)
    print("Rank:", rank)
    print("ID:", result["id"])
    print("RRF Score:", round(result["score"], 6))
    print("Text:", result["text"])

Rank: 1
ID: 0
RRF Score: 0.032522
Text: 持续降雨会增加土体含水率，并可能提高孔隙水压力。
Rank: 2
ID: 1
RRF Score: 0.032522
Text: 地下水位升高可能降低边坡稳定性。
Rank: 3
ID: 2
RRF Score: 0.031746
Text: 坡度是影响滑坡发生的重要地形因素。


In [20]:
def compare_three_methods(query):
    methods = {
        "BM25": bm25_search(query, 3),
        "Dense": dense_search(query, 3),
        "Hybrid": hybrid_search(query, 3, 8)
    }

    for name, results in methods.items():
        print(f"\n{'=' * 60}")
        print(name)

        for rank, item in enumerate(results, 1):
            print(
                f"{rank}. ID={item['id']} "
                f"Score={item['score']:.4f}"
            )
            print(item["text"])

In [21]:
compare_three_methods(
    "连续降水是否容易诱发坡体失稳？"
)

compare_three_methods(
    "WRF 4.5.1 是什么？"
)

compare_three_methods(
    "Noah-MP 是什么模型？"
)


BM25
1. ID=0 Score=0.0000
持续降雨会增加土体含水率，并可能提高孔隙水压力。
2. ID=1 Score=0.0000
地下水位升高可能降低边坡稳定性。
3. ID=2 Score=0.0000
坡度是影响滑坡发生的重要地形因素。

Dense
1. ID=1 Score=0.7650
地下水位升高可能降低边坡稳定性。
2. ID=0 Score=0.6658
持续降雨会增加土体含水率，并可能提高孔隙水压力。
3. ID=2 Score=0.6223
坡度是影响滑坡发生的重要地形因素。

Hybrid
1. ID=0 Score=0.0325
持续降雨会增加土体含水率，并可能提高孔隙水压力。
2. ID=1 Score=0.0325
地下水位升高可能降低边坡稳定性。
3. ID=2 Score=0.0317
坡度是影响滑坡发生的重要地形因素。

BM25
1. ID=5 Score=5.8424
WRF 4.5.1 是一种数值天气预报模型版本。
2. ID=0 Score=0.0000
持续降雨会增加土体含水率，并可能提高孔隙水压力。
3. ID=2 Score=0.0000
坡度是影响滑坡发生的重要地形因素。

Dense
1. ID=5 Score=0.7993
WRF 4.5.1 是一种数值天气预报模型版本。
2. ID=9 Score=0.4229
BGE-M3 支持多语言文本检索。
3. ID=11 Score=0.4219
Noah-MP 是一种陆面过程模型。

Hybrid
1. ID=5 Score=0.0328
WRF 4.5.1 是一种数值天气预报模型版本。
2. ID=1 Score=0.0303
地下水位升高可能降低边坡稳定性。
3. ID=7 Score=0.0303
RMSE 和 MAE 是常见的预测误差评价指标。

BM25
1. ID=11 Score=5.4208
Noah-MP 是一种陆面过程模型。
2. ID=8 Score=1.0671
PyTorch 可以用于深度学习模型训练。
3. ID=5 Score=0.9547
WRF 4.5.1 是一种数值天气预报模型版本。

Dense
1. ID=11 Score=0.8237
Noah-MP 是一种陆面过程模型。
2. ID=8 Score=0.49

In [22]:
hybrid_eval = evaluate_search(
    evaluation_dataset,
    hybrid_search,
    method_name="Hybrid-RRF",
    top_k=3
)

comparison_all = pd.concat(
    [
        bm25_eval,
        dense_eval,
        hybrid_eval
    ],
    ignore_index=True
)

summary_all = (
    comparison_all
    .groupby("method")[
        ["Hit@3", "Recall@3", "RR@3", "latency_ms"]
    ]
    .mean()
    .reset_index()
    .rename(columns={"RR@3": "MRR@3"})
)

display(summary_all.round(4))

,method,Hit@3,Recall@3,MRR@3,latency_ms
0,BM25,1.0,0.9375,1.0,0.2158
1,Dense,1.0,1.0000,1.0,6.5125
2,Hybrid-RRF,1.0,1.0000,1.0,6.7467
